[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/EN/Seminar_Notebooks/chapter11_seminar_notebook.ipynb)

---

# Chapter 11 Seminar: LLMs and Foundation Models for Time Series

**Course:** Time Series Analysis and Forecasting  
**Program:** Bachelor program, Faculty of Cybernetics, Statistics and Economic Informatics, Bucharest University of Economic Studies, Romania  
**Academic Year:** 2025-2026

---

## Seminar Objectives

In this practical seminar, you will:
1. Implement scaled dot-product self-attention from scratch and visualize attention weights
2. Explore sinusoidal positional encodings and understand how they encode relative position
3. Demonstrate time-series patching and analyze its effect on sequence length and complexity
4. Implement Chronos-style quantization and dequantization for continuous time series
5. Compare zero-shot foundation model forecasting against classical ARIMA baselines
6. Analyze LoRA (Low-Rank Adaptation) parameter efficiency for fine-tuning large models
7. Build a model selection decision framework for real-world forecasting scenarios
8. Investigate scaling laws in time-series foundation models
9. Ensemble classical and foundation model forecasts using variance-based weighting
10. Measure cross-domain transfer learning gaps with synthetic data

## Setup

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install yfinance scipy statsmodels -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from scipy import stats
from scipy.special import softmax
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# TSA course color palette
MainBlue = '#1A3A6E'
Crimson  = '#DC3545'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'

COLORS = {
    'blue': MainBlue, 'red': Crimson, 'green': Forest,
    'amber': Amber, 'orange': Orange, 'purple': Purple,
    'gray': '#666666', 'light_blue': '#5B8BD4'
}

plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.facecolor': 'none',
    'figure.facecolor': 'none',
    'savefig.facecolor': 'none',
    'legend.frameon': False
})

print('Setup complete. Libraries loaded.')

---
# Part I: Fundamentals (Year 3 Level)

**Objectives:** Build intuition for the core Transformer building blocks used in time-series foundation models — self-attention, positional encoding, patching, and tokenization via quantization.

## Exercise 1: Self-Attention by Hand

**Task:** Construct small $Q$, $K$, $V$ matrices (4 positions, $d_k = 3$) and compute scaled dot-product attention step by step:  
1. Compute the raw attention scores $QK^\top$  
2. Scale by $\sqrt{d_k}$  
3. Apply softmax row-wise to obtain attention weights  
4. Multiply by $V$ to get the output  
5. Visualize the attention weight matrix as a heatmap

**Scaled dot-product attention:**
$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

In [ ]:
# Step 1: Define Q, K, V matrices (4 positions, d_k = 3)
np.random.seed(42)
seq_len = 4
d_k = 3

Q = np.random.randn(seq_len, d_k).round(2)
K = np.random.randn(seq_len, d_k).round(2)
V = np.random.randn(seq_len, d_k).round(2)

print('Q (Query) matrix:')
print(Q)
print(f'\nK (Key) matrix:')
print(K)
print(f'\nV (Value) matrix:')
print(V)

# Step 2: Compute raw attention scores QK^T
raw_scores = Q @ K.T
print(f'\nRaw scores QK^T (shape {raw_scores.shape}):')
print(np.round(raw_scores, 4))

# Step 3: Scale by sqrt(d_k)
scale = np.sqrt(d_k)
scaled_scores = raw_scores / scale
print(f'\nScaled scores (divided by sqrt({d_k}) = {scale:.4f}):')
print(np.round(scaled_scores, 4))

# Step 4: Apply softmax row-wise
attention_weights = softmax(scaled_scores, axis=1)
print(f'\nAttention weights (softmax row-wise):')
print(np.round(attention_weights, 4))
print(f'Row sums (should be 1.0): {attention_weights.sum(axis=1).round(6)}')

# Step 5: Compute output = attention_weights @ V
output = attention_weights @ V
print(f'\nAttention output (shape {output.shape}):')
print(np.round(output, 4))

In [ ]:
# Visualize the attention weight matrix
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel 1: Raw scores
im0 = axes[0].imshow(raw_scores, cmap='Blues', aspect='auto')
axes[0].set_title('Raw Scores $QK^\\top$', fontweight='bold')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Query position')
axes[0].set_xticks(range(seq_len))
axes[0].set_yticks(range(seq_len))
for i in range(seq_len):
    for j in range(seq_len):
        axes[0].text(j, i, f'{raw_scores[i, j]:.2f}', ha='center', va='center',
                     color='white' if raw_scores[i, j] > raw_scores.mean() else 'black', fontsize=10)
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Panel 2: Attention weights (after softmax)
im1 = axes[1].imshow(attention_weights, cmap='Reds', aspect='auto', vmin=0, vmax=1)
axes[1].set_title('Attention Weights (after softmax)', fontweight='bold')
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Query position')
axes[1].set_xticks(range(seq_len))
axes[1].set_yticks(range(seq_len))
for i in range(seq_len):
    for j in range(seq_len):
        axes[1].text(j, i, f'{attention_weights[i, j]:.3f}', ha='center', va='center',
                     color='white' if attention_weights[i, j] > 0.4 else 'black', fontsize=10)
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# Panel 3: Bar chart — which position does each query attend to most?
max_attend = np.argmax(attention_weights, axis=1)
max_vals = np.max(attention_weights, axis=1)
bar_colors = [COLORS['blue'], COLORS['red'], COLORS['green'], COLORS['purple']]
bars = axes[2].bar(range(seq_len), max_vals, color=bar_colors, alpha=0.8, width=0.6)
for i, (pos, val) in enumerate(zip(max_attend, max_vals)):
    axes[2].text(i, val + 0.02, f'pos {pos}', ha='center', fontweight='bold', fontsize=10)
axes[2].set_title('Strongest Attention Target per Query', fontweight='bold')
axes[2].set_xlabel('Query position')
axes[2].set_ylabel('Max attention weight')
axes[2].set_xticks(range(seq_len))
axes[2].set_ylim(0, 1.0)
axes[2].spines[['top', 'right']].set_visible(False)
axes[2].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('Interpretation:')
for i in range(seq_len):
    print(f'  Query position {i} attends most to Key position {max_attend[i]} '
          f'(weight = {max_vals[i]:.3f})')

### Question 1.1

Which position attends most to position 0? Why?  
What would happen to the attention distribution if we did **not** scale by $\sqrt{d_k}$?

**YOUR ANSWER HERE**

## Exercise 2: Positional Encoding Exploration

**Task:** Implement sinusoidal positional encodings (Vaswani et al., 2017) and analyze their properties.  
1. Compute PE for $d_{\text{model}} = 64$ and $\text{max\_len} = 100$  
2. Visualize the encoding matrix as a heatmap  
3. Compute dot-product similarity between PE vectors at different positions  

**Sinusoidal positional encoding:**
$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \quad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    """Compute sinusoidal positional encoding matrix."""
    PE = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, np.newaxis]  # (max_len, 1)
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))  # (d_model/2,)
    PE[:, 0::2] = np.sin(position * div_term)  # even dimensions
    PE[:, 1::2] = np.cos(position * div_term)  # odd dimensions
    return PE

d_model = 64
max_len = 100
PE = sinusoidal_positional_encoding(max_len, d_model)

print(f'Positional encoding matrix shape: {PE.shape} (positions x dimensions)')
print(f'PE[0, :8] = {PE[0, :8].round(4)}')
print(f'PE[1, :8] = {PE[1, :8].round(4)}')

# Visualize the PE matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Full PE heatmap
im0 = axes[0].imshow(PE, cmap='RdBu_r', aspect='auto', interpolation='nearest')
axes[0].set_title('Sinusoidal Positional Encoding', fontweight='bold')
axes[0].set_xlabel('Dimension $i$')
axes[0].set_ylabel('Position $pos$')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Panel 2: Selected dimensions
dims_to_plot = [0, 1, 4, 5, 16, 17, 32, 33]
dim_colors = [MainBlue, Crimson, Forest, Amber, Orange, Purple, '#666666', '#5B8BD4']
for dim, color in zip(dims_to_plot, dim_colors):
    axes[1].plot(range(max_len), PE[:, dim], linewidth=1.2, alpha=0.8, color=color,
                 label=f'dim {dim}')
axes[1].set_title('PE Across Positions (Selected Dimensions)', fontweight='bold')
axes[1].set_xlabel('Position $pos$')
axes[1].set_ylabel('PE value')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=4, frameon=False, fontsize=9)
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('Key observation: low-index dimensions oscillate rapidly (high frequency),')
print('while high-index dimensions oscillate slowly (low frequency).')
print('This creates a unique "fingerprint" for each position.')

In [ ]:
# Compute dot-product similarity between PE vectors
similarity = PE @ PE.T  # (max_len, max_len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Full similarity matrix
im0 = axes[0].imshow(similarity, cmap='RdBu_r', aspect='auto')
axes[0].set_title('PE Dot-Product Similarity Matrix', fontweight='bold')
axes[0].set_xlabel('Position $j$')
axes[0].set_ylabel('Position $i$')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Panel 2: Similarity relative to position 0
ref_positions = [0, 10, 50]
ref_colors = [MainBlue, Crimson, Forest]
for ref, color in zip(ref_positions, ref_colors):
    sim_vec = similarity[ref, :]
    axes[1].plot(range(max_len), sim_vec, linewidth=1.5, color=color,
                 label=f'Similarity to pos {ref}')
axes[1].set_title('Dot-Product Similarity vs Distance', fontweight='bold')
axes[1].set_xlabel('Position $j$')
axes[1].set_ylabel('$PE_i \\cdot PE_j$')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  - The similarity is highest on the diagonal (position with itself).')
print('  - Similarity decays with distance, encoding relative position information.')
print('  - The sinusoidal structure creates a smooth, periodic decay pattern.')
print(f'  - PE(0) . PE(1) = {similarity[0, 1]:.4f}')
print(f'  - PE(0) . PE(10) = {similarity[0, 10]:.4f}')
print(f'  - PE(0) . PE(50) = {similarity[0, 50]:.4f}')

### Question 2.1

What happens to PE similarity as positions get farther apart?  
Why is this property essential for the Transformer to learn temporal dependencies?

**YOUR ANSWER HERE**

## Exercise 3: Patching Demo

**Task:** Demonstrate time-series patching — the key innovation in PatchTST and other modern TS Transformers.  
1. Generate a sine wave with 512 time steps  
2. Divide it into non-overlapping patches of size $P = 16, 32, 64$  
3. Visualize the patches and calculate the complexity reduction  

**Patching** groups consecutive time steps into tokens, reducing the sequence length from $L$ to $L/P$ and enabling the Transformer to capture local semantic information within each patch.

In [ ]:
# Generate a sine wave with 512 points
L = 512
t = np.arange(L)
signal = np.sin(2 * np.pi * t / 64) + 0.3 * np.sin(2 * np.pi * t / 16) + 0.1 * np.random.randn(L)

# Patch sizes to compare
patch_sizes = [16, 32, 64]
patch_colors = [MainBlue, Crimson, Forest]

fig, axes = plt.subplots(len(patch_sizes) + 1, 1, figsize=(14, 10))

# Panel 0: Original signal
axes[0].plot(t, signal, color=COLORS['gray'], linewidth=0.8)
axes[0].set_title(f'Original Signal ($L = {L}$ time steps)', fontweight='bold')
axes[0].set_ylabel('Value')
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].set_facecolor('none')

print(f'{"Patch Size (P)":>15s} {"Num Patches":>12s} {"Seq Reduction":>15s} {"Attn Complexity":>20s}')
print('-' * 65)

for idx, (P, color) in enumerate(zip(patch_sizes, patch_colors)):
    n_patches = L // P
    seq_reduction = L / n_patches
    attn_original = L * L  # O(L^2)
    attn_patched = n_patches * n_patches  # O((L/P)^2)
    complexity_ratio = attn_patched / attn_original
    
    print(f'{P:>15d} {n_patches:>12d} {seq_reduction:>14.0f}x {complexity_ratio:>19.4f} ({complexity_ratio*100:.2f}%)')
    
    ax = axes[idx + 1]
    for p in range(n_patches):
        start = p * P
        end = start + P
        patch_t = t[start:end]
        patch_val = signal[start:end]
        alpha = 0.6 if p % 2 == 0 else 0.3
        ax.fill_between(patch_t, patch_val, alpha=alpha, color=color)
        ax.plot(patch_t, patch_val, color=color, linewidth=0.5)
        if P >= 32:
            ax.axvline(start, color=color, linewidth=0.3, alpha=0.5)
    
    ax.set_title(f'Patching with $P = {P}$  ({n_patches} patches, '
                 f'attention complexity = {complexity_ratio*100:.1f}%)', fontweight='bold')
    ax.set_ylabel('Value')
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('none')

axes[-1].set_xlabel('Time step')
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

In [ ]:
# Complexity reduction plot
P_range = np.arange(1, 129)
n_patches_range = L / P_range
complexity_range = (n_patches_range ** 2) / (L ** 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Number of patches vs P
axes[0].plot(P_range, n_patches_range, color=MainBlue, linewidth=2)
for P, color in zip(patch_sizes, patch_colors):
    axes[0].scatter([P], [L / P], color=color, s=100, zorder=5)
    axes[0].annotate(f'P={P}\n({L // P} patches)', (P, L / P),
                     textcoords='offset points', xytext=(15, 5), fontsize=9,
                     arrowprops=dict(arrowstyle='->', color=color), color=color)
axes[0].set_title('Number of Patches vs Patch Size', fontweight='bold')
axes[0].set_xlabel('Patch size $P$')
axes[0].set_ylabel('Number of patches $L/P$')
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].set_facecolor('none')

# Panel 2: Attention complexity reduction
axes[1].plot(P_range, complexity_range * 100, color=Crimson, linewidth=2)
for P, color in zip(patch_sizes, patch_colors):
    comp = ((L / P) ** 2) / (L ** 2) * 100
    axes[1].scatter([P], [comp], color=color, s=100, zorder=5)
    axes[1].annotate(f'P={P}\n({comp:.2f}%)', (P, comp),
                     textcoords='offset points', xytext=(15, 5), fontsize=9,
                     arrowprops=dict(arrowstyle='->', color=color), color=color)
axes[1].set_title('Attention Complexity as % of Original', fontweight='bold')
axes[1].set_xlabel('Patch size $P$')
axes[1].set_ylabel('Complexity (% of $O(L^2)$)')
axes[1].set_yscale('log')
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('Trade-off summary:')
print('  Larger P -> fewer patches -> much lower attention complexity')
print('  Larger P -> each patch spans more time -> may lose fine-grained patterns')
print('  Smaller P -> more patches -> higher resolution but higher cost')

### Question 3.1

What is the trade-off of larger patch sizes?  
If your time series has important patterns at the 5-step scale, what patch size would you choose and why?

**YOUR ANSWER HERE**

## Exercise 4: Chronos-Style Quantization

**Task:** Implement the tokenization strategy used by Amazon Chronos:  
1. Download EUR/RON exchange rate data using yfinance  
2. Normalize: $\tilde{x}_t = x_t \,/\, |\bar{x}|$  
3. Quantize using the CDF of the standard normal: $\text{bin}(\tilde{x}_t) = \lfloor B \cdot \Phi(\tilde{x}_t) \rfloor$  
4. Dequantize by inverting the process  
5. Measure reconstruction error and plot original vs reconstructed

This approach converts continuous time-series values into discrete tokens that can be processed by a language model.

In [ ]:
# Download EUR/RON exchange rate data
eurron = yf.download('EURRON=X', start='2020-01-01', end='2024-12-31', progress=False)
eurron = eurron[['Close']].dropna()
eurron.columns = ['Price']

# Use log-returns for quantization
eurron['Return'] = np.log(eurron['Price']).diff()
returns = eurron['Return'].dropna().values

print(f'EUR/RON data: {len(eurron)} observations')
print(f'Returns: mean = {returns.mean():.6f}, std = {returns.std():.6f}')

# Step 1: Normalize
mean_abs = np.abs(returns.mean())
if mean_abs < 1e-10:
    mean_abs = returns.std()  # fallback if mean is near zero
x_tilde = returns / mean_abs
print(f'\nNormalization factor |mean(x)| = {mean_abs:.8f}')
print(f'Normalized range: [{x_tilde.min():.2f}, {x_tilde.max():.2f}]')

# Step 2: Quantize using normal CDF into B bins
B = 4096
cdf_vals = stats.norm.cdf(x_tilde)
bins = np.floor(B * cdf_vals).astype(int)
bins = np.clip(bins, 0, B - 1)

print(f'\nQuantization to B = {B} bins:')
print(f'  Bin range: [{bins.min()}, {bins.max()}]')
print(f'  Unique bins used: {len(np.unique(bins))}')

# Step 3: Dequantize
# Map bin center back through inverse CDF
bin_centers = (bins + 0.5) / B
x_tilde_reconstructed = stats.norm.ppf(bin_centers)
returns_reconstructed = x_tilde_reconstructed * mean_abs

# Step 4: Measure reconstruction error
recon_error = returns - returns_reconstructed
rmse = np.sqrt(np.mean(recon_error ** 2))
mae = np.mean(np.abs(recon_error))
max_error = np.max(np.abs(recon_error))

print(f'\nReconstruction quality:')
print(f'  RMSE:      {rmse:.8f}')
print(f'  MAE:       {mae:.8f}')
print(f'  Max error: {max_error:.8f}')

In [ ]:
# Visualize quantization results
dates = eurron.index[1:]  # skip first NaN return

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Panel 1: Original vs Reconstructed returns
axes[0, 0].plot(dates[:200], returns[:200], color=MainBlue, linewidth=0.8, alpha=0.8,
                label='Original')
axes[0, 0].plot(dates[:200], returns_reconstructed[:200], color=Crimson, linewidth=0.8,
                alpha=0.8, linestyle='--', label='Reconstructed')
axes[0, 0].set_title('Original vs Reconstructed Returns (first 200)', fontweight='bold')
axes[0, 0].set_ylabel('Log-return')
axes[0, 0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
axes[0, 0].spines[['top', 'right']].set_visible(False)
axes[0, 0].set_facecolor('none')

# Panel 2: Reconstruction error
axes[0, 1].plot(dates, recon_error, color=Forest, linewidth=0.5, alpha=0.7)
axes[0, 1].axhline(0, color=COLORS['gray'], linestyle='--', linewidth=0.8)
axes[0, 1].set_title('Reconstruction Error', fontweight='bold')
axes[0, 1].set_ylabel('Error')
axes[0, 1].spines[['top', 'right']].set_visible(False)
axes[0, 1].set_facecolor('none')

# Panel 3: Histogram of bin assignments
axes[1, 0].hist(bins, bins=100, color=Purple, alpha=0.7, density=True)
axes[1, 0].set_title(f'Distribution of Bin Assignments ($B = {B}$)', fontweight='bold')
axes[1, 0].set_xlabel('Bin index')
axes[1, 0].set_ylabel('Density')
axes[1, 0].spines[['top', 'right']].set_visible(False)
axes[1, 0].set_facecolor('none')

# Panel 4: RMSE vs number of bins
bin_counts = [32, 64, 128, 256, 512, 1024, 2048, 4096, 8192]
rmse_vs_bins = []
for b in bin_counts:
    cdf_b = stats.norm.cdf(x_tilde)
    bins_b = np.clip(np.floor(b * cdf_b).astype(int), 0, b - 1)
    centers_b = (bins_b + 0.5) / b
    recon_b = stats.norm.ppf(centers_b) * mean_abs
    rmse_b = np.sqrt(np.mean((returns - recon_b) ** 2))
    rmse_vs_bins.append(rmse_b)

axes[1, 1].plot(bin_counts, rmse_vs_bins, 'o-', color=MainBlue, linewidth=2, markersize=8)
axes[1, 1].axvline(4096, color=Crimson, linestyle='--', linewidth=1.5, alpha=0.7,
                    label='Chronos default ($B=4096$)')
axes[1, 1].set_title('Reconstruction RMSE vs Number of Bins', fontweight='bold')
axes[1, 1].set_xlabel('Number of bins $B$')
axes[1, 1].set_ylabel('RMSE')
axes[1, 1].set_xscale('log', base=2)
axes[1, 1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False)
axes[1, 1].spines[['top', 'right']].set_visible(False)
axes[1, 1].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('Conclusion:')
print('  - Quantization with B=4096 bins preserves time-series information with minimal loss.')
print('  - RMSE drops sharply up to ~512 bins, then diminishing returns set in.')
print('  - The CDF-based mapping concentrates bins where data is densest (near the center).')

### Question 4.1

How does the number of bins affect reconstruction quality?  
Why does Chronos use a CDF-based mapping instead of uniform quantization?

**YOUR ANSWER HERE**

---
# Part II: Advanced (Master Level)

**Objectives:** Compare zero-shot foundation model forecasts with classical baselines, analyze the parameter efficiency of LoRA fine-tuning, and develop a structured model selection framework for real-world scenarios.

## Exercise 5: Zero-Shot Forecasting Comparison

**Task:** Compare a classical ARIMA(1,1,1) forecast against a simulated zero-shot foundation model forecast on EUR/RON data.  
1. Download EUR/RON from 2020 to 2024  
2. Use the last 60 observations as a test set  
3. Fit ARIMA(1,1,1) on the training set and produce 60-step forecasts  
4. Simulate a Chronos-style zero-shot forecast (using the quantization pipeline + a simple local-level model as a stand-in, since the full Chronos model requires a GPU)  
5. Compare RMSE, MAE, and visualize predictions

**Note:** The "zero-shot" forecast here is simulated to demonstrate the evaluation framework. In practice, you would use the `chronos-forecasting` library with a pre-trained model.

In [ ]:
# Download EUR/RON data
eurron_full = yf.download('EURRON=X', start='2020-01-01', end='2024-12-31', progress=False)
eurron_full = eurron_full[['Close']].dropna()
eurron_full.columns = ['Price']

# Train/test split
test_size = 60
train = eurron_full.iloc[:-test_size]
test = eurron_full.iloc[-test_size:]

print(f'Training set: {len(train)} observations ({train.index[0].strftime("%Y-%m-%d")} to '
      f'{train.index[-1].strftime("%Y-%m-%d")})')
print(f'Test set:     {test_size} observations ({test.index[0].strftime("%Y-%m-%d")} to '
      f'{test.index[-1].strftime("%Y-%m-%d")})')

# Method 1: ARIMA(1,1,1)
arima_model = ARIMA(train['Price'], order=(1, 1, 1))
arima_fit = arima_model.fit()
arima_forecast = arima_fit.forecast(steps=test_size)

print(f'\nARIMA(1,1,1) fitted. AIC = {arima_fit.aic:.2f}')

# Method 2: Simulated zero-shot forecast
# In practice, this would be: from chronos import ChronosPipeline
# Here we simulate it with a random-walk-with-drift + mean-reversion approach
# to approximate what a pre-trained model might produce
try:
    # Attempt to import chronos (will likely fail in most environments)
    from chronos import ChronosPipeline
    pipeline = ChronosPipeline.from_pretrained('amazon/chronos-t5-small')
    import torch
    context = torch.tensor(train['Price'].values, dtype=torch.float32).unsqueeze(0)
    chronos_forecast_raw = pipeline.predict(context, prediction_length=test_size)
    zs_forecast = chronos_forecast_raw[0].median(dim=0).values.numpy()
    zs_method = 'Chronos-T5-Small (actual)'
except Exception:
    # Simulated zero-shot: exponential smoothing of recent history
    # This mimics a foundation model that learns local patterns from context
    recent = train['Price'].values[-120:]
    trend = np.polyfit(np.arange(len(recent)), recent, deg=1)
    last_price = train['Price'].values[-1]
    noise_std = np.std(np.diff(recent)) * 0.5
    np.random.seed(123)
    zs_forecast_vals = [last_price]
    for i in range(test_size):
        drift = trend[0] * 0.5  # dampened trend
        mean_rev = 0.02 * (recent.mean() - zs_forecast_vals[-1])  # slight mean reversion
        next_val = zs_forecast_vals[-1] + drift + mean_rev + np.random.randn() * noise_std
        zs_forecast_vals.append(next_val)
    zs_forecast = np.array(zs_forecast_vals[1:])
    zs_method = 'Simulated Zero-Shot (Chronos-style)'
    print(f'\nNote: Chronos not available. Using simulated zero-shot forecast.')

# Compute metrics
actual = test['Price'].values

def forecast_metrics(actual, predicted, name):
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    mae = np.mean(np.abs(actual - predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    return {'Method': name, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

m_arima = forecast_metrics(actual, arima_forecast.values, 'ARIMA(1,1,1)')
m_zs = forecast_metrics(actual, zs_forecast, zs_method)

print(f'\n{"Metric":>10s} {"ARIMA(1,1,1)":>15s} {"Zero-Shot":>15s}')
print('-' * 42)
for metric in ['RMSE', 'MAE', 'MAPE']:
    unit = '%' if metric == 'MAPE' else ''
    print(f'{metric:>10s} {m_arima[metric]:>14.4f}{unit} {m_zs[metric]:>14.4f}{unit}')

In [ ]:
# Visualize forecasts
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Panel 1: Price with forecasts
axes[0].plot(train.index[-120:], train['Price'].values[-120:], color=COLORS['gray'],
             linewidth=1, label='Training data')
axes[0].plot(test.index, actual, color=MainBlue, linewidth=2, label='Actual')
axes[0].plot(test.index, arima_forecast.values, color=Crimson, linewidth=1.5,
             linestyle='--', label='ARIMA(1,1,1)')
axes[0].plot(test.index, zs_forecast, color=Forest, linewidth=1.5,
             linestyle='-.', label=zs_method)
axes[0].axvline(test.index[0], color=COLORS['gray'], linestyle=':', linewidth=1, alpha=0.5)
axes[0].set_title('EUR/RON Forecast Comparison', fontweight='bold')
axes[0].set_ylabel('Price')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False, fontsize=9)
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].set_facecolor('none')

# Panel 2: Cumulative absolute error
cae_arima = np.cumsum(np.abs(actual - arima_forecast.values))
cae_zs = np.cumsum(np.abs(actual - zs_forecast))
axes[1].plot(test.index, cae_arima, color=Crimson, linewidth=2, label='ARIMA(1,1,1)')
axes[1].plot(test.index, cae_zs, color=Forest, linewidth=2, label=zs_method)
axes[1].set_title('Cumulative Absolute Error', fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Cumulative |error|')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  - ARIMA excels when the DGP is close to a linear model with Gaussian noise.')
print('  - Zero-shot models can capture nonlinear patterns without task-specific training.')
print('  - The comparison depends heavily on the data characteristics and forecast horizon.')

### Question 5.1

Under what conditions would you expect zero-shot forecasting to outperform ARIMA?  
What are the key advantages and limitations of each approach?

**YOUR ANSWER HERE**

## Exercise 6: LoRA Parameter Analysis

**Task:** Analyze the parameter efficiency of Low-Rank Adaptation (LoRA) for fine-tuning a Transformer.  
1. Given model dimensions $d = 512$, $h = 8$ heads, $L = 6$ layers, compute the total attention parameters  
2. Compute LoRA parameters for ranks $r = 4, 8, 16, 32$  
3. Plot the percentage of trainable parameters vs rank  

**LoRA** freezes the pre-trained weights $W$ and adds a low-rank decomposition $\Delta W = BA$ where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times d}$, so the trainable parameters per weight matrix are $2dr$ instead of $d^2$.

In [ ]:
# Model dimensions
d = 512       # model dimension
h = 8         # number of attention heads
n_layers = 6  # number of transformer layers
d_k = d // h  # key/query dimension per head

# Attention parameters per layer: W_Q, W_K, W_V, W_O (each d x d)
params_per_attn_matrix = d * d
attn_matrices_per_layer = 4  # Q, K, V, O
attn_params_per_layer = attn_matrices_per_layer * params_per_attn_matrix
total_attn_params = n_layers * attn_params_per_layer

# Also count FFN parameters: typically 2 layers of d -> 4d -> d
ffn_params_per_layer = 2 * d * (4 * d)  # W1: d->4d, W2: 4d->d
total_ffn_params = n_layers * ffn_params_per_layer
total_params = total_attn_params + total_ffn_params

print('Model Architecture Summary')
print('=' * 55)
print(f'  Model dimension (d):           {d}')
print(f'  Attention heads (h):           {h}')
print(f'  Head dimension (d_k):          {d_k}')
print(f'  Transformer layers:            {n_layers}')
print(f'  Attention matrices per layer:  {attn_matrices_per_layer} (Q, K, V, O)')
print(f'\nParameter Count:')
print(f'  Attention params per layer:    {attn_params_per_layer:>12,}')
print(f'  FFN params per layer:          {ffn_params_per_layer:>12,}')
print(f'  Total attention params:        {total_attn_params:>12,}')
print(f'  Total FFN params:              {total_ffn_params:>12,}')
print(f'  Total model params:            {total_params:>12,} ({total_params/1e6:.1f}M)')

# LoRA parameters for different ranks
ranks = [1, 2, 4, 8, 16, 32, 64, 128]

print(f'\n{"Rank r":>8s} {"LoRA params/matrix":>20s} {"Total LoRA params":>20s} '
      f'{"% of full":>12s} {"Compression":>12s}')
print('-' * 75)

lora_results = []
for r in ranks:
    # LoRA params per matrix: B (d x r) + A (r x d) = 2*d*r
    lora_per_matrix = 2 * d * r
    # Apply LoRA to Q, V matrices in each layer (common practice)
    n_lora_matrices = 2 * n_layers  # Q and V in each layer
    total_lora = n_lora_matrices * lora_per_matrix
    pct = total_lora / total_params * 100
    compression = total_params / total_lora
    lora_results.append({'rank': r, 'total_lora': total_lora, 'pct': pct,
                         'compression': compression})
    print(f'{r:>8d} {lora_per_matrix:>20,} {total_lora:>20,} {pct:>11.2f}% {compression:>11.1f}x')

lora_df = pd.DataFrame(lora_results)

In [ ]:
# Visualize LoRA efficiency
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Trainable parameters vs rank
axes[0].bar(range(len(ranks)), [r['total_lora'] for r in lora_results],
            color=MainBlue, alpha=0.8, width=0.6)
axes[0].axhline(total_params, color=Crimson, linestyle='--', linewidth=1.5,
                label=f'Full fine-tuning ({total_params:,})')
axes[0].set_xticks(range(len(ranks)))
axes[0].set_xticklabels([str(r) for r in ranks])
axes[0].set_title('Trainable Parameters vs LoRA Rank', fontweight='bold')
axes[0].set_xlabel('LoRA rank $r$')
axes[0].set_ylabel('Number of parameters')
axes[0].set_yscale('log')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False)
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].set_facecolor('none')

# Panel 2: Percentage of total params
axes[1].plot(ranks, [r['pct'] for r in lora_results], 'o-', color=Forest,
             linewidth=2, markersize=8)
axes[1].axhline(100, color=Crimson, linestyle='--', linewidth=1.5, alpha=0.7,
                label='Full fine-tuning (100%)')
for r_data in lora_results:
    if r_data['rank'] in [4, 16, 64]:
        axes[1].annotate(f'{r_data["pct"]:.1f}%', (r_data['rank'], r_data['pct']),
                         textcoords='offset points', xytext=(10, 5), fontsize=9,
                         color=Forest, fontweight='bold')
axes[1].set_title('% of Trainable Parameters vs Rank', fontweight='bold')
axes[1].set_xlabel('LoRA rank $r$')
axes[1].set_ylabel('% of full model parameters')
axes[1].set_xscale('log', base=2)
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False)
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].set_facecolor('none')

# Panel 3: Compression ratio
axes[2].plot(ranks, [r['compression'] for r in lora_results], 's-', color=Purple,
             linewidth=2, markersize=8)
for r_data in lora_results:
    if r_data['rank'] in [4, 16, 64]:
        axes[2].annotate(f'{r_data["compression"]:.0f}x', (r_data['rank'], r_data['compression']),
                         textcoords='offset points', xytext=(10, 5), fontsize=9,
                         color=Purple, fontweight='bold')
axes[2].set_title('Compression Ratio vs Rank', fontweight='bold')
axes[2].set_xlabel('LoRA rank $r$')
axes[2].set_ylabel('Compression ratio (full / LoRA)')
axes[2].set_xscale('log', base=2)
axes[2].set_yscale('log')
axes[2].spines[['top', 'right']].set_visible(False)
axes[2].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('Key insights:')
print(f'  - At rank r=4, LoRA uses only {lora_results[2]["pct"]:.2f}% of the full parameters')
print(f'  - At rank r=16, LoRA uses {lora_results[4]["pct"]:.2f}% of the full parameters')
print(f'  - Even at r=128, LoRA uses {lora_results[-1]["pct"]:.2f}% of full fine-tuning')
print(f'  - In practice, r=4 to r=16 often achieves near-full fine-tuning performance')

### Question 6.1

At what rank does LoRA approach full fine-tuning in terms of parameter count?  
Why is LoRA particularly attractive for adapting large foundation models to specific time-series domains?

**YOUR ANSWER HERE**

## Exercise 7: Model Selection Decision Tree

**Task:** Given 5 real-world forecasting scenarios, recommend the most appropriate model and justify your choice.  
Then create a visual decision flowchart for selecting between classical, deep learning, and foundation model approaches.

**Scenarios:**
1. **Cold-start:** A new product with only 30 data points, no similar historical products  
2. **Data-rich:** 20 years of hourly energy consumption data for a single building  
3. **Real-time:** Sub-second financial tick data requiring <10ms inference  
4. **Interpretability:** A regulator requires full explanation of GDP growth forecasts  
5. **Multivariate:** 50 correlated sensor streams from an industrial plant

In [ ]:
# Define the 5 scenarios and recommendations
scenarios = [
    {
        'name': 'Cold-Start',
        'desc': 'New product, 30 data points, no similar history',
        'data_size': 'Very Small',
        'latency': 'Flexible',
        'interpretability': 'Low',
        'recommendation': 'Zero-Shot FM\n(Chronos)',
        'reasoning': 'Too little data for classical estimation. FM provides\n'
                     'priors from massive pre-training corpus.',
        'color': Forest
    },
    {
        'name': 'Data-Rich',
        'desc': '20 years hourly energy data, single building',
        'data_size': 'Very Large',
        'latency': 'Flexible',
        'interpretability': 'Low',
        'recommendation': 'Fine-tuned FM\nor Deep Learning',
        'reasoning': 'Ample data to fine-tune. PatchTST or fine-tuned\n'
                     'Chronos can capture complex seasonality.',
        'color': MainBlue
    },
    {
        'name': 'Real-Time',
        'desc': 'Sub-second tick data, <10ms latency',
        'data_size': 'Large',
        'latency': 'Critical',
        'interpretability': 'Low',
        'recommendation': 'Classical\n(ARMA/ETS)',
        'reasoning': 'FM inference too slow. Classical models are lightweight\n'
                     'and can update online in microseconds.',
        'color': Crimson
    },
    {
        'name': 'Interpretability',
        'desc': 'Central bank needs interpretable GDP forecasts',
        'data_size': 'Small-Medium',
        'latency': 'Flexible',
        'interpretability': 'Critical',
        'recommendation': 'Classical\n(VAR/ECM)',
        'reasoning': 'Regulators need coefficient interpretation and\n'
                     'impulse responses. Black-box models are not acceptable.',
        'color': Crimson
    },
    {
        'name': 'Multivariate',
        'desc': '50 correlated sensor streams, industrial plant',
        'data_size': 'Large',
        'latency': 'Moderate',
        'interpretability': 'Low',
        'recommendation': 'Channel-Indep. FM\n+ VAR ensemble',
        'reasoning': 'PatchTST channel-independent approach scales well.\n'
                     'Ensemble with VAR for cross-series dependencies.',
        'color': Purple
    }
]

# Print scenario analysis
print('Model Selection Analysis')
print('=' * 90)
for i, s in enumerate(scenarios, 1):
    print(f'\nScenario {i}: {s["name"]}')
    print(f'  Description:      {s["desc"]}')
    print(f'  Data size:        {s["data_size"]}')
    print(f'  Latency needs:    {s["latency"]}')
    print(f'  Interpretability: {s["interpretability"]}')
    print(f'  Recommendation:   {s["recommendation"].replace(chr(10), " ")}')
    print(f'  Reasoning:        {s["reasoning"].replace(chr(10), " ")}')

In [ ]:
# Create a decision flowchart
fig, ax = plt.subplots(1, 1, figsize=(14, 10))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Helper functions for drawing
def draw_box(ax, x, y, w, h, text, color, fontsize=9):
    rect = mpatches.FancyBboxPatch((x - w/2, y - h/2), w, h,
                                    boxstyle='round,pad=0.15', linewidth=1.5,
                                    edgecolor=color, facecolor=color, alpha=0.15)
    ax.add_patch(rect)
    rect_border = mpatches.FancyBboxPatch((x - w/2, y - h/2), w, h,
                                          boxstyle='round,pad=0.15', linewidth=1.5,
                                          edgecolor=color, facecolor='none')
    ax.add_patch(rect_border)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            fontweight='bold', color=color)

def draw_diamond(ax, x, y, w, h, text, color, fontsize=9):
    diamond = plt.Polygon([(x, y + h/2), (x + w/2, y), (x, y - h/2), (x - w/2, y)],
                          closed=True, fill=True, facecolor=color, alpha=0.12,
                          edgecolor=color, linewidth=1.5)
    ax.add_patch(diamond)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            fontweight='bold', color=color)

def draw_arrow(ax, x1, y1, x2, y2, label='', color='gray'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, linewidth=1.5))
    if label:
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mx + 0.15, my + 0.1, label, fontsize=8, color=color, fontstyle='italic')

# Root node
draw_diamond(ax, 5, 9.2, 3.2, 0.9, 'Need interpretability?', MainBlue, 9)

# Left branch: Yes -> Classical
draw_arrow(ax, 3.4, 9.2, 2.0, 8.0, 'Yes', Forest)
draw_box(ax, 2.0, 7.6, 2.4, 0.7, 'Classical\n(ARIMA, VAR, ECM)', Crimson, 9)

# Right branch: No -> next question
draw_arrow(ax, 6.6, 9.2, 8.0, 8.0, 'No', Crimson)
draw_diamond(ax, 7.5, 7.5, 3.0, 0.9, 'Latency < 10ms?', MainBlue, 9)

# Right-Left: Yes -> Classical
draw_arrow(ax, 6.0, 7.5, 4.5, 6.3, 'Yes', Forest)
draw_box(ax, 4.5, 5.9, 2.4, 0.7, 'Lightweight Classical\n(ARMA, ETS, online)', Crimson, 9)

# Right-Right: No -> data question
draw_arrow(ax, 9.0, 7.5, 8.0, 6.0, 'No', Crimson)
draw_diamond(ax, 8.0, 5.7, 2.6, 0.9, 'Enough data\nfor training?', MainBlue, 8)

# Large data -> Fine-tune / Deep Learning
draw_arrow(ax, 9.3, 5.7, 9.0, 4.3, 'Yes', Forest)
draw_box(ax, 8.5, 3.9, 2.5, 0.7, 'Fine-tuned FM\nor Deep Learning', Forest, 9)

# Small data -> zero-shot
draw_arrow(ax, 6.7, 5.7, 5.5, 4.3, 'No', Crimson)
draw_diamond(ax, 5.5, 3.9, 2.8, 0.9, 'Multivariate?', MainBlue, 9)

# Univariate -> zero-shot FM
draw_arrow(ax, 4.1, 3.9, 3.0, 2.7, 'No', Crimson)
draw_box(ax, 3.0, 2.3, 2.4, 0.7, 'Zero-Shot FM\n(Chronos, TimesFM)', Forest, 9)

# Multivariate -> ensemble
draw_arrow(ax, 6.9, 3.9, 7.5, 2.7, 'Yes', Forest)
draw_box(ax, 7.5, 2.3, 2.8, 0.7, 'Channel-Indep. FM\n+ VAR ensemble', Purple, 9)

# Title
ax.text(5, 0.8, 'Model Selection Decision Tree for Time-Series Forecasting',
        ha='center', va='center', fontsize=13, fontweight='bold', color=MainBlue)

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

### Question 7.1

A central bank needs interpretable 1-year GDP forecasts. Which approach would you recommend and why?  
Could a foundation model ever satisfy this requirement? Under what conditions?

**YOUR ANSWER HERE**

---
# Part III: PhD Extension

**Objectives:** Investigate scaling laws, explore forecast ensembling strategies, and measure cross-domain transfer learning performance.

## Exercise 8: Scaling Analysis

**Task:** Analyze the scaling behavior of time-series foundation models.  
1. Plot parameter count vs benchmark score (WQL on GluonTS) for Chronos model variants  
2. Overlay the NLP Chinchilla scaling curve for comparison  
3. Compute a diminishing-returns metric to assess where adding more parameters yields marginal gains

**Chronos variants** (from Ansari et al., 2024):
- Chronos-T5-Mini: 20M params  
- Chronos-T5-Small: 46M params  
- Chronos-T5-Base: 200M params  
- Chronos-T5-Large: 710M params

In [ ]:
# Chronos scaling data (approximate from paper)
chronos_models = {
    'Mini': {'params_M': 20, 'wql': 0.485},
    'Small': {'params_M': 46, 'wql': 0.470},
    'Base': {'params_M': 200, 'wql': 0.445},
    'Large': {'params_M': 710, 'wql': 0.430}
}

names = list(chronos_models.keys())
params_M = np.array([chronos_models[n]['params_M'] for n in names])
wql_scores = np.array([chronos_models[n]['wql'] for n in names])

# NLP scaling curve (Chinchilla-style): loss ~ C * N^(-alpha)
# Typical NLP alpha ~ 0.076 (Hoffmann et al., 2022)
nlp_params = np.logspace(1, 3.5, 100)  # 10M to ~3B
nlp_loss = 0.65 * (nlp_params) ** (-0.076)  # approximate Chinchilla scaling

# Fit a power law to Chronos data
log_params = np.log(params_M)
log_wql = np.log(wql_scores)
slope, intercept = np.polyfit(log_params, log_wql, 1)
alpha_ts = -slope
ts_fitted = np.exp(intercept) * nlp_params ** slope

# Diminishing returns: marginal improvement per 100M params
marginal_improvements = []
for i in range(1, len(names)):
    delta_params = params_M[i] - params_M[i-1]
    delta_wql = wql_scores[i-1] - wql_scores[i]  # improvement = decrease in WQL
    marginal = delta_wql / (delta_params / 100)  # improvement per 100M params
    marginal_improvements.append({
        'from': names[i-1], 'to': names[i],
        'delta_params': delta_params, 'delta_wql': delta_wql,
        'marginal_per_100M': marginal
    })

print('Chronos Scaling Analysis')
print('=' * 60)
print(f'{"Model":>10s} {"Params (M)":>12s} {"WQL Score":>12s}')
print('-' * 36)
for name in names:
    print(f'{name:>10s} {chronos_models[name]["params_M"]:>12d} {chronos_models[name]["wql"]:>12.3f}')

print(f'\nFitted scaling exponent (alpha_TS): {alpha_ts:.4f}')
print(f'NLP scaling exponent (alpha_NLP):   0.0760')
print(f'\nDiminishing Returns Analysis:')
print(f'{"Transition":>20s} {"Extra Params":>15s} {"WQL Improvement":>16s} {"Gain/100M":>12s}')
print('-' * 65)
for m in marginal_improvements:
    print(f'{m["from"] + " -> " + m["to"]:>20s} {m["delta_params"]:>12d}M '
          f'{m["delta_wql"]:>15.3f} {m["marginal_per_100M"]:>11.4f}')

In [ ]:
# Visualize scaling curves
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Chronos scaling
axes[0].scatter(params_M, wql_scores, color=MainBlue, s=120, zorder=5)
for name, pm, wql in zip(names, params_M, wql_scores):
    axes[0].annotate(f'  {name}\n  ({pm}M)', (pm, wql), fontsize=9, color=MainBlue)
axes[0].plot(nlp_params, ts_fitted, color=MainBlue, linewidth=1.5, linestyle='--',
             alpha=0.5, label=f'Power law fit ($\\alpha$={alpha_ts:.3f})')
axes[0].set_title('Chronos: Params vs WQL Score', fontweight='bold')
axes[0].set_xlabel('Parameters (millions)')
axes[0].set_ylabel('WQL Score (lower is better)')
axes[0].set_xscale('log')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False)
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].set_facecolor('none')

# Panel 2: TS vs NLP scaling comparison
# Normalize both to start at 1.0 for comparison
ts_norm = ts_fitted / ts_fitted[0]
nlp_norm = nlp_loss / nlp_loss[0]
axes[1].plot(nlp_params, ts_norm, color=MainBlue, linewidth=2,
             label=f'Time Series ($\\alpha$={alpha_ts:.3f})')
axes[1].plot(nlp_params, nlp_norm, color=Crimson, linewidth=2, linestyle='--',
             label=f'NLP Chinchilla ($\\alpha$=0.076)')
axes[1].set_title('Scaling Comparison: TS vs NLP', fontweight='bold')
axes[1].set_xlabel('Parameters (millions)')
axes[1].set_ylabel('Normalized loss (relative to smallest)')
axes[1].set_xscale('log')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False)
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].set_facecolor('none')

# Panel 3: Marginal improvement (diminishing returns)
transitions = [f'{m["from"]}\n->{m["to"]}' for m in marginal_improvements]
marginals = [m['marginal_per_100M'] for m in marginal_improvements]
bar_colors = [Forest, Amber, Crimson]
bars = axes[2].bar(range(len(transitions)), marginals, color=bar_colors, alpha=0.8, width=0.6)
for bar, val in zip(bars, marginals):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', fontweight='bold', fontsize=9)
axes[2].set_xticks(range(len(transitions)))
axes[2].set_xticklabels(transitions, fontsize=9)
axes[2].set_title('Diminishing Returns:\nWQL Gain per 100M Parameters', fontweight='bold')
axes[2].set_ylabel('WQL improvement / 100M params')
axes[2].spines[['top', 'right']].set_visible(False)
axes[2].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('Key findings:')
print(f'  - TS scaling exponent ({alpha_ts:.3f}) is in a similar range to NLP (0.076).')
print('  - Marginal improvement decreases sharply: Mini->Small gains the most per parameter.')
print('  - Base->Large adds 510M params but only improves WQL by ~0.015.')
print('  - This suggests that DATA QUALITY and DIVERSITY may matter more than raw scale.')

### Question 8.1

What additional data would most improve time-series foundation models?  
Is the scaling bottleneck in parameters, data, or compute? How does this differ from NLP?

**YOUR ANSWER HERE**

## Exercise 9: Ensemble FM + Classical

**Task:** Combine an ARIMA forecast with a simulated foundation model forecast using different weighting schemes:  
1. Simple average (equal weights)  
2. Inverse-variance weighting (based on recent forecast errors)  
3. Compare ensemble performance vs individual models  

**Inverse-variance weighting:** $w_i = \frac{1/\sigma_i^2}{\sum_j 1/\sigma_j^2}$

In [ ]:
# Use EUR/RON data from Exercise 5
# We already have: arima_forecast, zs_forecast, actual, test

# Method 1: Simple average
ensemble_avg = 0.5 * arima_forecast.values + 0.5 * zs_forecast

# Method 2: Inverse-variance weighting
# Estimate variance from the first 20 test points, apply to the rest
calibration_window = 20

# Rolling inverse-variance weights
ensemble_iv = np.zeros(test_size)
weights_arima_history = []
weights_zs_history = []

for t_idx in range(test_size):
    if t_idx < calibration_window:
        # Before we have enough data, use equal weights
        w_arima = 0.5
        w_zs = 0.5
    else:
        # Compute variance from recent errors
        recent_err_arima = actual[t_idx - calibration_window:t_idx] - arima_forecast.values[t_idx - calibration_window:t_idx]
        recent_err_zs = actual[t_idx - calibration_window:t_idx] - zs_forecast[t_idx - calibration_window:t_idx]
        var_arima = np.var(recent_err_arima) + 1e-10
        var_zs = np.var(recent_err_zs) + 1e-10
        inv_var_arima = 1.0 / var_arima
        inv_var_zs = 1.0 / var_zs
        total_inv = inv_var_arima + inv_var_zs
        w_arima = inv_var_arima / total_inv
        w_zs = inv_var_zs / total_inv
    
    ensemble_iv[t_idx] = w_arima * arima_forecast.values[t_idx] + w_zs * zs_forecast[t_idx]
    weights_arima_history.append(w_arima)
    weights_zs_history.append(w_zs)

# Compute metrics for all methods
methods = {
    'ARIMA(1,1,1)': arima_forecast.values,
    'Zero-Shot FM': zs_forecast,
    'Simple Average': ensemble_avg,
    'Inv-Variance': ensemble_iv
}

print('Forecast Comparison')
print('=' * 65)
print(f'{"Method":>20s} {"RMSE":>10s} {"MAE":>10s} {"MAPE (%)":>10s}')
print('-' * 52)
for name, pred in methods.items():
    rmse = np.sqrt(np.mean((actual - pred) ** 2))
    mae = np.mean(np.abs(actual - pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    print(f'{name:>20s} {rmse:>10.4f} {mae:>10.4f} {mape:>10.2f}')

In [ ]:
# Visualize ensemble results
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Panel 1: All forecasts
axes[0, 0].plot(test.index, actual, color=COLORS['gray'], linewidth=2, label='Actual')
axes[0, 0].plot(test.index, arima_forecast.values, color=Crimson, linewidth=1, alpha=0.7,
                linestyle='--', label='ARIMA')
axes[0, 0].plot(test.index, zs_forecast, color=Forest, linewidth=1, alpha=0.7,
                linestyle='-.', label='Zero-Shot')
axes[0, 0].plot(test.index, ensemble_iv, color=MainBlue, linewidth=2,
                label='Inv-Variance Ensemble')
axes[0, 0].set_title('All Forecasts', fontweight='bold')
axes[0, 0].set_ylabel('Price')
axes[0, 0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False,
                   fontsize=9)
axes[0, 0].spines[['top', 'right']].set_visible(False)
axes[0, 0].set_facecolor('none')

# Panel 2: Adaptive weights over time
axes[0, 1].plot(test.index, weights_arima_history, color=Crimson, linewidth=1.5,
                label='ARIMA weight')
axes[0, 1].plot(test.index, weights_zs_history, color=Forest, linewidth=1.5,
                label='Zero-Shot weight')
axes[0, 1].axhline(0.5, color=COLORS['gray'], linestyle='--', linewidth=0.8)
axes[0, 1].axvline(test.index[calibration_window], color=Amber, linestyle=':',
                    linewidth=1.5, label=f'Calibration end (t={calibration_window})')
axes[0, 1].set_title('Adaptive Weights (Inverse-Variance)', fontweight='bold')
axes[0, 1].set_ylabel('Weight')
axes[0, 1].set_ylim(0, 1)
axes[0, 1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False,
                   fontsize=9)
axes[0, 1].spines[['top', 'right']].set_visible(False)
axes[0, 1].set_facecolor('none')

# Panel 3: Absolute errors comparison
ae_arima = np.abs(actual - arima_forecast.values)
ae_zs = np.abs(actual - zs_forecast)
ae_ens = np.abs(actual - ensemble_iv)
axes[1, 0].plot(test.index, ae_arima, color=Crimson, linewidth=0.8, alpha=0.7, label='ARIMA')
axes[1, 0].plot(test.index, ae_zs, color=Forest, linewidth=0.8, alpha=0.7, label='Zero-Shot')
axes[1, 0].plot(test.index, ae_ens, color=MainBlue, linewidth=1.5, label='Ensemble')
axes[1, 0].set_title('Absolute Forecast Errors', fontweight='bold')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('|Error|')
axes[1, 0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False,
                   fontsize=9)
axes[1, 0].spines[['top', 'right']].set_visible(False)
axes[1, 0].set_facecolor('none')

# Panel 4: RMSE bar comparison
method_names = list(methods.keys())
rmse_vals = [np.sqrt(np.mean((actual - methods[m]) ** 2)) for m in method_names]
bar_colors = [Crimson, Forest, Amber, MainBlue]
bars = axes[1, 1].bar(range(len(method_names)), rmse_vals, color=bar_colors, alpha=0.8, width=0.6)
for bar, val in zip(bars, rmse_vals):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                     f'{val:.4f}', ha='center', fontweight='bold', fontsize=9)
axes[1, 1].set_xticks(range(len(method_names)))
axes[1, 1].set_xticklabels(method_names, fontsize=9, rotation=15)
axes[1, 1].set_title('RMSE Comparison', fontweight='bold')
axes[1, 1].set_ylabel('RMSE')
axes[1, 1].spines[['top', 'right']].set_visible(False)
axes[1, 1].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  - Ensembling generally reduces forecast variance ("wisdom of crowds").')
print('  - Inverse-variance weighting adapts to which model is currently more accurate.')
print('  - Ensembling can HURT when models are highly correlated and one is much worse.')

### Question 9.1

When does ensembling hurt rather than help?  
What conditions on model correlation and individual accuracy determine ensemble benefit?

**YOUR ANSWER HERE**

## Exercise 10: Transfer Learning Experiment

**Task:** Measure how well temporal patterns transfer across domains using synthetic data.  
1. Generate synthetic "energy" data (strong daily/weekly seasonality + trend)  
2. Generate synthetic "retail" data (weekly seasonality + holiday spikes + noise)  
3. Train a simple model (exponential smoothing) on one domain, evaluate on the other  
4. Measure the transfer gap (performance degradation when applying across domains)  

This simulates the key challenge faced by foundation models: how well do patterns learned from one domain generalize to another?

In [ ]:
np.random.seed(42)

# Generate synthetic energy data (365 days)
n_days = 365
t_days = np.arange(n_days)

# Energy: daily cycle + weekly pattern + slow trend + noise
energy_trend = 100 + 0.05 * t_days
energy_daily = 15 * np.sin(2 * np.pi * t_days / 1)   # daily oscillation
energy_weekly = 8 * np.sin(2 * np.pi * t_days / 7)    # weekly pattern
energy_annual = 20 * np.sin(2 * np.pi * (t_days - 30) / 365)  # seasonal (heating/cooling)
energy_noise = np.random.randn(n_days) * 5
energy = energy_trend + energy_daily + energy_weekly + energy_annual + energy_noise

# Retail: weekly seasonality + holiday spikes + trend + noise
retail_trend = 50 + 0.03 * t_days
retail_weekly = 12 * np.sin(2 * np.pi * t_days / 7 + 1.5)  # different phase
retail_annual = 10 * np.sin(2 * np.pi * (t_days - 90) / 365)  # different seasonal phase
# Holiday spikes (simulated at days 90, 180, 330, 355)
retail_holidays = np.zeros(n_days)
for holiday in [90, 180, 330, 355]:
    mask = np.exp(-0.5 * ((t_days - holiday) / 3) ** 2)
    retail_holidays += 30 * mask
retail_noise = np.random.randn(n_days) * 4
retail = retail_trend + retail_weekly + retail_annual + retail_holidays + retail_noise

# Visualize both domains
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(t_days, energy, color=MainBlue, linewidth=0.8)
axes[0].set_title('Synthetic Energy Consumption Data', fontweight='bold')
axes[0].set_ylabel('Energy (kWh)')
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].set_facecolor('none')

axes[1].plot(t_days, retail, color=Crimson, linewidth=0.8)
axes[1].set_title('Synthetic Retail Sales Data', fontweight='bold')
axes[1].set_ylabel('Sales (units)')
axes[1].set_xlabel('Day')
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print(f'Energy: mean={energy.mean():.1f}, std={energy.std():.1f}')
print(f'Retail: mean={retail.mean():.1f}, std={retail.std():.1f}')

In [ ]:
# Train/test split for both domains
train_size = 300
test_len = n_days - train_size

energy_train, energy_test = energy[:train_size], energy[train_size:]
retail_train, retail_test = retail[:train_size], retail[train_size:]

# Simple model: Exponential Smoothing with additive seasonality
from statsmodels.tsa.holtwinters import ExponentialSmoothing

def fit_and_forecast(train_data, test_len, seasonal_period=7):
    """Fit Holt-Winters and return forecast."""
    model = ExponentialSmoothing(
        train_data, trend='add', seasonal='add',
        seasonal_periods=seasonal_period
    ).fit(optimized=True)
    forecast = model.forecast(test_len)
    # Also extract the learned parameters for transfer
    return forecast, model

# In-domain performance
energy_fc, energy_model = fit_and_forecast(energy_train, test_len)
retail_fc, retail_model = fit_and_forecast(retail_train, test_len)

# Cross-domain transfer: apply energy model's smoothing parameters to retail
# Simulate transfer by using energy-learned level/trend/seasonal components
# and evaluating on the other domain

# Transfer approach: normalize both domains, train on one, predict the other
def normalize(data):
    return (data - data.mean()) / data.std(), data.mean(), data.std()

def denormalize(data, mean, std):
    return data * std + mean

# Normalize training sets
energy_train_n, e_mean, e_std = normalize(energy_train)
retail_train_n, r_mean, r_std = normalize(retail_train)

# Train on normalized energy, predict normalized test
energy_model_n = ExponentialSmoothing(
    energy_train_n, trend='add', seasonal='add', seasonal_periods=7
).fit(optimized=True)

# Transfer: use energy model to forecast retail (in normalized space)
# Re-fit with energy parameters but on retail data
energy_fc_on_retail_n = energy_model_n.forecast(test_len)
energy_fc_on_retail = denormalize(energy_fc_on_retail_n, r_mean, r_std)

# Similarly, train on retail and transfer to energy
retail_model_n = ExponentialSmoothing(
    retail_train_n, trend='add', seasonal='add', seasonal_periods=7
).fit(optimized=True)
retail_fc_on_energy_n = retail_model_n.forecast(test_len)
retail_fc_on_energy = denormalize(retail_fc_on_energy_n, e_mean, e_std)

# Compute RMSE for all scenarios
def rmse(actual, predicted):
    return np.sqrt(np.mean((actual - predicted) ** 2))

results = {
    'Energy -> Energy (in-domain)': rmse(energy_test, energy_fc),
    'Retail -> Retail (in-domain)': rmse(retail_test, retail_fc),
    'Energy -> Retail (transfer)': rmse(retail_test, energy_fc_on_retail),
    'Retail -> Energy (transfer)': rmse(energy_test, retail_fc_on_energy)
}

print('Transfer Learning Results')
print('=' * 55)
print(f'{"Scenario":>35s} {"RMSE":>10s}')
print('-' * 47)
for name, val in results.items():
    print(f'{name:>35s} {val:>10.4f}')

# Transfer gaps
gap_energy = results['Retail -> Energy (transfer)'] / results['Energy -> Energy (in-domain)'] - 1
gap_retail = results['Energy -> Retail (transfer)'] / results['Retail -> Retail (in-domain)'] - 1
print(f'\nTransfer gap (Energy domain): {gap_energy*100:+.1f}% RMSE increase')
print(f'Transfer gap (Retail domain): {gap_retail*100:+.1f}% RMSE increase')

In [ ]:
# Visualize transfer results
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

t_test = t_days[train_size:]

# Panel 1: Energy in-domain
axes[0, 0].plot(t_test, energy_test, color=COLORS['gray'], linewidth=1, label='Actual')
axes[0, 0].plot(t_test, energy_fc, color=MainBlue, linewidth=1.5, label='In-domain forecast')
axes[0, 0].set_title(f'Energy (In-Domain) - RMSE: {results["Energy -> Energy (in-domain)"]:.2f}',
                      fontweight='bold')
axes[0, 0].set_ylabel('Energy (kWh)')
axes[0, 0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
axes[0, 0].spines[['top', 'right']].set_visible(False)
axes[0, 0].set_facecolor('none')

# Panel 2: Energy with transfer from retail
axes[0, 1].plot(t_test, energy_test, color=COLORS['gray'], linewidth=1, label='Actual')
axes[0, 1].plot(t_test, retail_fc_on_energy, color=Crimson, linewidth=1.5,
                label='Transfer forecast (from retail)')
axes[0, 1].set_title(f'Energy (Transfer from Retail) - RMSE: {results["Retail -> Energy (transfer)"]:.2f}',
                      fontweight='bold')
axes[0, 1].set_ylabel('Energy (kWh)')
axes[0, 1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
axes[0, 1].spines[['top', 'right']].set_visible(False)
axes[0, 1].set_facecolor('none')

# Panel 3: Retail in-domain
axes[1, 0].plot(t_test, retail_test, color=COLORS['gray'], linewidth=1, label='Actual')
axes[1, 0].plot(t_test, retail_fc, color=Forest, linewidth=1.5, label='In-domain forecast')
axes[1, 0].set_title(f'Retail (In-Domain) - RMSE: {results["Retail -> Retail (in-domain)"]:.2f}',
                      fontweight='bold')
axes[1, 0].set_xlabel('Day')
axes[1, 0].set_ylabel('Sales (units)')
axes[1, 0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
axes[1, 0].spines[['top', 'right']].set_visible(False)
axes[1, 0].set_facecolor('none')

# Panel 4: Retail with transfer from energy
axes[1, 1].plot(t_test, retail_test, color=COLORS['gray'], linewidth=1, label='Actual')
axes[1, 1].plot(t_test, energy_fc_on_retail, color=Orange, linewidth=1.5,
                label='Transfer forecast (from energy)')
axes[1, 1].set_title(f'Retail (Transfer from Energy) - RMSE: {results["Energy -> Retail (transfer)"]:.2f}',
                      fontweight='bold')
axes[1, 1].set_xlabel('Day')
axes[1, 1].set_ylabel('Sales (units)')
axes[1, 1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
axes[1, 1].spines[['top', 'right']].set_visible(False)
axes[1, 1].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze which temporal patterns transfer well
# Decompose both series to extract trend, seasonal, residual
from statsmodels.tsa.seasonal import seasonal_decompose

energy_decomp = seasonal_decompose(energy, model='additive', period=7)
retail_decomp = seasonal_decompose(retail, model='additive', period=7)

# Compute correlation between components
# Trim NaNs from decomposition
valid = ~(np.isnan(energy_decomp.trend) | np.isnan(retail_decomp.trend))

trend_corr = np.corrcoef(energy_decomp.trend[valid], retail_decomp.trend[valid])[0, 1]
seasonal_corr = np.corrcoef(energy_decomp.seasonal[valid], retail_decomp.seasonal[valid])[0, 1]
resid_corr = np.corrcoef(energy_decomp.resid[valid], retail_decomp.resid[valid])[0, 1]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

components = ['Trend', 'Seasonal', 'Residual']
correlations = [trend_corr, seasonal_corr, resid_corr]
bar_colors = [MainBlue, Forest, Crimson]
bars = ax.bar(components, correlations, color=bar_colors, alpha=0.8, width=0.5)
for bar, val in zip(bars, correlations):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontweight='bold', fontsize=11)
ax.axhline(0, color=COLORS['gray'], linestyle='--', linewidth=0.8)
ax.set_title('Cross-Domain Component Correlation\n(Energy vs Retail)', fontweight='bold')
ax.set_ylabel('Pearson correlation')
ax.set_ylim(-1, 1)
ax.spines[['top', 'right']].set_visible(False)
ax.set_facecolor('none')
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('Component Transferability:')
print(f'  Trend correlation:    {trend_corr:+.3f} {"(transfers well)" if abs(trend_corr) > 0.7 else "(weak transfer)"}')
print(f'  Seasonal correlation: {seasonal_corr:+.3f} {"(transfers well)" if abs(seasonal_corr) > 0.5 else "(weak transfer)"}')
print(f'  Residual correlation: {resid_corr:+.3f} {"(transfers well)" if abs(resid_corr) > 0.3 else "(no transfer expected)"}')
print()
print('Key finding: Trend components (smooth, long-range) transfer best across domains.')
print('Seasonal patterns are domain-specific and transfer poorly.')
print('Residuals (noise) do not transfer, as expected.')
print()
print('Implication for foundation models:')
print('  FMs that learn general trend/growth patterns can transfer across domains.')
print('  Domain-specific seasonality requires either fine-tuning or domain adaptation.')

### Question 10.1

What temporal patterns transfer best across domains?  
How can foundation models leverage transferable patterns while adapting to domain-specific seasonality?

**YOUR ANSWER HERE**

---
## Summary: Chapter 11 Seminar

| Part | Exercises | Key Concepts |
|------|-----------|-------------|
| I Fundamentals | 1–4 | Attention, Positional Encoding, Patching, Quantization |
| II Master | 5–7 | Zero-Shot Forecasting, LoRA, Model Selection |
| III PhD | 8–10 | Scaling Laws, Ensembles, Transfer Learning |

### Key Takeaways

- **Self-attention** computes weighted averages of Value vectors, with weights determined by Query-Key similarity scaled by $\sqrt{d_k}$
- **Sinusoidal positional encodings** provide unique position fingerprints with smooth distance-dependent similarity decay
- **Patching** reduces Transformer complexity from $O(L^2)$ to $O((L/P)^2)$ while preserving local semantic information
- **Chronos-style quantization** converts continuous time series into discrete tokens via CDF mapping, enabling language model architectures
- **Zero-shot forecasting** excels in cold-start settings where classical models lack sufficient training data
- **LoRA** achieves near-full fine-tuning performance with only 1-5% of the parameters, making FM adaptation practical
- **Model selection** depends on data availability, latency requirements, interpretability needs, and whether the task is univariate or multivariate
- **Scaling laws** for TS foundation models show diminishing returns, suggesting data diversity matters more than raw parameter count
- **Ensembling** classical + FM forecasts often improves performance through variance reduction, but fails when models are highly correlated
- **Transfer learning** works best for smooth trend patterns; domain-specific seasonality requires fine-tuning or adaptation